### Model Context Protocol

### Server 1: `mathserver.py`

**FastMCP**: A high-level Python library that simplifies building MCP servers. By placing `@mcp.tool()` above standard Python functions, it automatically extracts docstrings and type annotations to build standard JSON schemas for the LLM.

**Transport (`stdio`)**: The transport is the communication channel connecting the client to the server. `stdio` uses the operating system's standard input (`stdin`) and standard output (`stdout`) streams. The agent client spawns `mathserver.py` as a background child process and sends JSON-RPC messages back and forth through standard console pipes—no network port or IP needed

Fast, secure, and run locally on your laptop without needing an internet connection, network ports, or IP addresses.

In [8]:
%%writefile mathserver.py
from mcp.server.fastmcp import FastMCP

mcp=FastMCP("Math")

@mcp.tool()
def add(a:int,b:int)->int:
    """Add two numbers."""
    return a+b

@mcp.tool()
def multiple(a:int,b:int)->int:
    """Multiply two numbers."""
    return a*b

if __name__=="__main__":
    mcp.run(transport="stdio")

Overwriting mathserver.py


### Server 2: `weather.py`

**Transport (`streamable-http`)**: Unlike `stdio`, which communicates via local OS pipes, `streamable-http` launches an ASGI web server (powered by Uvicorn and Starlette) bound to a network address (e.g., `http://127.0.0.1:8000`).

The tool runs as an independent web server (using Uvicorn) listening on an address like http://localhost:8000. The AI talks to it over the network using standard web requests.

Great when tools live on a remote server, inside a Docker container, or are shared among multiple users.

**Session Management & SSE**: It handles incoming HTTP requests and streams responses back using Server-Sent Events (SSE). This setup allows remote hosting (cloud VMs, Docker containers, multi-tenant services) so multiple distributed clients can query the same server.

In [9]:
%%writefile weather.py
from mcp.server.fastmcp import FastMCP

mcp=FastMCP("Weather")

@mcp.tool()
async def get_weather(location:str)->str:
    """Get the weather location."""
    return f"It's always raining in {location}"

if __name__=="__main__":
    mcp.run(transport="streamable-http")


Overwriting weather.py


### Starting the Background HTTP Server

Because `weather.py` acts as a network service, we launch it as a background process so our client can connect to `http://localhost:8000/mcp`.

In [10]:
import subprocess
import time
import sys

if 'weather_process' in locals():
    weather_process.terminate()

weather_process=subprocess.Popen([sys.executable, "weather.py"])
print("Started weather.py web server. Waiting 3 seconds for Uvicorn to bind")

time.sleep(3)

print("Weather server ready")

Started weather.py web server. Waiting 3 seconds for Uvicorn to bind
Weather server ready


### Orchestration: `client.py` (Multi-Server Agent)

**`MultiServerMCPClient`**: An adapter that unifies multiple disparate MCP tool sources. It manages both subprocess execution (`stdio` for `mathserver.py`) and netawork sessions (`streamable-http` for `weather.py`), consolidating all registered functions into a single list of LangChain tools via `client.get_tools()`.

In [11]:
import os
import sys
import asyncio
from dotenv import load_dotenv
from langchain_mcp_adapters.client import MultiServerMCPClient
from langgraph.prebuilt import create_react_agent
from langchain.chat_models import init_chat_model

# Fix Windows Jupyter fileno() issue: restore underlying OS stderr for subprocess creation
if hasattr(sys, "__stderr__") and sys.__stderr__ is not None:
    sys.stderr = sys.__stderr__

load_dotenv()

os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")

model=init_chat_model("groq:openai/gpt-oss-20b")

async def run_agent():
    # Initialize MultiServerMCPClient connecting to both servers
    client=MultiServerMCPClient({
        "math": {
            "command": sys.executable,  # Uses the current Python runtime
            "args": ["mathserver.py"],
            "transport": "stdio",
        },
        "weather": {
            "url": "http://localhost:8000/mcp", # HTTP endpoint
            "transport": "streamable_http",
        }
    })

    # Fetch all tools dynamically from both servers
    tools=await client.get_tools()
    print(f"Loaded {len(tools)} tools: {[t.name for t in tools]}")

    # Construct the ReAct agent graph with the model and tools
    agent=create_react_agent(model, tools)

    # Query 1: Arithmetic Decomposition
    print("\nQuery 1: Math")
    math_response=await agent.ainvoke({
        "messages":[{"role": "user","content": "what's (3 + 5) x 12?"}]
    })
    print("Math response:", math_response["messages"][-1].content)

    # Query 2: Remote HTTP Tool Query
    print("\nQuery 2: Weather")
    weather_response=await agent.ainvoke({
        "messages": [{"role": "user","content": "what is the weather in California?"}]
    })
    print("Weather response:",weather_response["messages"][-1].content)

# Execute the async agent function
await run_agent()


UnsupportedOperation: fileno

HTTP Request: POST http://localhost:8000/mcp "HTTP/1.1 200 OK"
Received session ID: 81ec1bc47c724ef7a3e978deecdd2aba
Negotiated protocol version: 2025-11-25
HTTP Request: POST http://localhost:8000/mcp "HTTP/1.1 202 Accepted"
HTTP Request: GET http://localhost:8000/mcp "HTTP/1.1 200 OK"
HTTP Request: POST http://localhost:8000/mcp "HTTP/1.1 200 OK"
HTTP Request: DELETE http://localhost:8000/mcp "HTTP/1.1 200 OK"


### Cleanup

Terminate the background HTTP weather server process once execution completes.

In [12]:
if 'weather_process' in locals():
    weather_process.terminate()
    print("Terminated weather server background process.")

Terminated weather server background process.
